# 1. Gerar base de exemplo

> ### ⚠️ Dados totalmente fictícios
> Nada aqui vem de uma base real. Nomes, CPFs, e-mails, telefones, cidades e valores são **sorteados por este próprio código**. Os CPFs têm onze dígitos aleatórios e **não passam na validação do dígito verificador**, ou seja, são inválidos por construção e não podem corresponder a nenhuma pessoa existente. Qualquer semelhança com dados reais seria coincidência estatística.

Cria um cadastro de clientes com os problemas de qualidade que aparecem em praticamente todo sistema de pequena empresa.

Ao final, o arquivo `cadastro_clientes.csv` fica salvo e serve de entrada para o notebook 2.

**Problemas incluídos de propósito:** campos em branco, cadastros duplicados, espaços invisíveis, números como texto em formatos misturados, datas em três padrões, mesma categoria escrita de várias formas, CPF truncado, email inválido e valores negativos.

In [ ]:
import random
from datetime import date, timedelta
import pandas as pd

SEED = 7          # trava o sorteio, para o resultado ser sempre o mesmo
N = 600           # quantidade de cadastros
ARQUIVO = "cadastro_clientes.csv"

random.seed(SEED)

## Catálogos

In [ ]:
NOMES = ["Ana", "Bruno", "Carla", "Diego", "Elisa", "Fabio", "Gabriela",
         "Heitor", "Isabel", "Joao", "Karina", "Lucas", "Marina", "Nelson",
         "Olivia", "Paulo", "Renata", "Sergio", "Tatiana", "Vinicius"]

SOBRENOMES = ["Silva", "Santos", "Oliveira", "Souza", "Costa", "Pereira",
              "Almeida", "Ferreira", "Rodrigues", "Martins", "Barbosa", "Rocha"]

# a mesma cidade escrita de varios jeitos, que e o problema mais comum
# em cadastro preenchido a mao
CIDADES = {
    "Porto Alegre": ["Porto Alegre", "porto alegre", "PORTO ALEGRE", "P. Alegre", "Poa"],
    "Santa Maria": ["Santa Maria", "santa maria", "STA MARIA", "Sta. Maria"],
    "Passo Fundo": ["Passo Fundo", "passo fundo", "PASSO FUNDO"],
    "Ijui": ["Ijui", "Ijui", "IJUI", "ijui"],
    "Santo Angelo": ["Santo Angelo", "Santo Angelo", "STO ANGELO", "santo angelo"],
}

CANAIS = ["Loja", "loja", "LOJA", "Site", "site", "Whatsapp", "WhatsApp",
          "whatsapp", "Indicacao", "Indicacao", "Telefone"]

## Funções que sujam os dados

Cada função abaixo estraga um tipo de informação, imitando o que acontece quando o preenchimento é manual ou quando o sistema exporta de módulos diferentes.

In [ ]:
def gerar_cpf():
    """
    Onze digitos sorteados, sem calculo do digito verificador.

    Isso e proposital. Gerar CPF valido criaria numeros que poderiam
    pertencer a pessoas reais, e nao ha motivo para correr esse risco
    numa base de demonstracao.
    """
    return "".join(str(random.randint(0, 9)) for _ in range(11))


def sujar_cpf(c):
    f = random.choices(["mascara", "limpo", "curto"], [0.5, 0.4, 0.1])[0]
    if f == "mascara":
        return f"{c[:3]}.{c[3:6]}.{c[6:9]}-{c[9:]}"
    if f == "curto":
        return c[:9]                     # CPF truncado, nao valida
    return c


def sujar_telefone():
    ddd = random.choice(["51", "54", "55", "11"])
    n = "9" + "".join(str(random.randint(0, 9)) for _ in range(8))
    f = random.choices(["completo", "limpo", "sem_ddd", "com_ramal"],
                       [0.4, 0.3, 0.2, 0.1])[0]
    if f == "completo":
        return f"({ddd}) {n[:5]}-{n[5:]}"
    if f == "limpo":
        return f"{ddd}{n}"
    if f == "sem_ddd":
        return f"{n[:5]}-{n[5:]}"
    return f"{ddd} {n} ramal 22"


def sujar_email(nome, sobrenome):
    base = f"{nome}.{sobrenome}".lower()
    dom = random.choice(["gmail.com", "hotmail.com", "outlook.com", "empresa.com.br"])
    f = random.choices(["ok", "sem_arroba", "espaco", "vazio"], [0.85, 0.06, 0.05, 0.04])[0]
    if f == "sem_arroba":
        return f"{base}.{dom}"
    if f == "espaco":
        return f" {base}@{dom} "
    if f == "vazio":
        return ""
    return f"{base}@{dom}"


def sujar_data(d):
    f = random.choices(["br", "iso", "curta"], [0.6, 0.3, 0.1])[0]
    if f == "br":
        return d.strftime("%d/%m/%Y")
    if f == "iso":
        return d.strftime("%Y-%m-%d")
    return d.strftime("%d-%m-%y")


def sujar_valor(v):
    f = random.choices(["virgula", "ponto", "moeda"], [0.6, 0.3, 0.1])[0]
    if f == "virgula":
        return f"{v:.2f}".replace(".", ",")
    if f == "moeda":
        return f"R$ {v:.2f}".replace(".", ",")
    return f"{v:.2f}"

## Montagem dos cadastros

In [ ]:
hoje = date(2025, 12, 31)
linhas = []

for i in range(N):
    nome = random.choice(NOMES)
    sobrenome = random.choice(SOBRENOMES)
    cidade = random.choice(list(CIDADES))
    cadastro = hoje - timedelta(days=random.randint(30, 1800))

    linhas.append({
        "id_cliente": 1000 + i,
        "nome": random.choice([
            f"{nome} {sobrenome}",
            f"  {nome} {sobrenome}  ",           # espacos sobrando
            f"{nome.upper()} {sobrenome.upper()}",
            f"{nome} {sobrenome}",
        ]),
        "cpf": sujar_cpf(gerar_cpf()),
        "email": sujar_email(nome, sobrenome),
        "telefone": sujar_telefone(),
        "cidade": random.choice(CIDADES[cidade]),
        "uf": random.choice(["RS", "rs", "R.S.", "RS "]),
        "canal_origem": random.choice(CANAIS),
        "data_cadastro": sujar_data(cadastro),
        "limite_credito": sujar_valor(round(random.uniform(500, 15000), 2)),
        "total_comprado": sujar_valor(round(random.uniform(0, 90000), 2)),
        "ativo": random.choice(["S", "N", "s", "n", "Sim", "Nao", "1", "0"]),
    })

df = pd.DataFrame(linhas)
print(f"{len(df)} cadastros montados")

## Últimos defeitos

Aplicados depois da tabela pronta, porque são falhas que acontecem na operação e não no preenchimento de cada campo.

In [ ]:
# campos deixados em branco
for coluna, taxa in [("email", 0.09), ("telefone", 0.14),
                     ("cidade", 0.05), ("limite_credito", 0.07)]:
    idx = random.sample(range(len(df)), k=int(len(df) * taxa))
    df.loc[idx, coluna] = ""

# mesma pessoa cadastrada duas vezes, com identificador diferente
dup = df.sample(frac=0.06, random_state=SEED).copy()
dup["id_cliente"] = range(90000, 90000 + len(dup))
df = pd.concat([df, dup], ignore_index=True)

# valores impossiveis
idx = random.sample(range(len(df)), k=12)
df.loc[idx, "total_comprado"] = "-1500,00"

# coluna praticamente vazia, resquicio de sistema antigo
df["obs_sistema"] = ""
idx = random.sample(range(len(df)), k=5)
df.loc[idx, "obs_sistema"] = "migrado 2019"

df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f"{len(df)} linhas depois das duplicatas")

## Gravação

O arquivo sai em `latin-1` com ponto e vírgula, que é como sistema brasileiro antigo costuma exportar.

In [ ]:
df.to_csv(ARQUIVO, sep=";", index=False, encoding="latin-1", errors="replace")
print(f"{ARQUIVO} gravado com {len(df)} linhas e {len(df.columns)} colunas")
df.head(5)